In [1]:
# ===== C1 clone + clock =====
import time, subprocess
NB_START=time.time()
subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && git clone -q https://github.com/CIawevy/FreeFine.git", shell=True, check=True); print("cloned")

cloned


In [2]:
%%bash
# ===== C2 freefine_env (generation) =====
set -e
pip install -q --root-user-action=ignore uv; uv python install 3.10.13
V=/kaggle/temp/freefine_env; PY=$V/bin/python
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/temp/FreeFine
uv pip install --python "$PY" -r requirements.txt || { grep -v '^xformers' requirements.txt>/tmp/r.txt; uv pip install --python "$PY" -r /tmp/r.txt; uv pip install --python "$PY" xformers; }
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70"
"$PY" -c "import torch,diffusers,xformers; print('freefine_env OK',torch.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 66.1 MB/s eta 0:00:00
freefine_env OK 2.1.1+cu121


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.43s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/freefine_env
Activate with: source /kaggle/temp/freefine_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/freefine_env
Resolved 18 packages in 1.02s
 Downloaded torchvision
 Downloaded pillow
 Downloaded triton
 Downloaded networkx
 Downloaded numpy
 Downloaded sympy
 Downloaded torch
Prepared 18 packages in 36.04s
Installed 18 packages in 277ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.29.0
 + fsspec==2026.4.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.2.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.15.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/temp/freefine_en

In [3]:
%%bash
# ===== C3 metric_env (evaluation) =====
set -e
V=/kaggle/temp/metric_env; PY=$V/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null||true; done
echo "metric_env OK"

metric_env OK


Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 825ms
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded pillow
 Downloaded nvidia-nvjitlink-cu12
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded networkx
 Downloaded numpy
 Downloaded nvidia-curand-cu12
 Downloaded triton
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded sympy
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 45.52s
Installed 27 packages in 261ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4

In [4]:
# ===== C4 patch metrics (args.3d, SD-2.1 mirror, SEEDED MD) + model.py START_LAYER fix =====
import pathlib, re
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"; mp.write_text(mp.read_text().replace("args.3d","getattr(args,'3d')"))
for f in [mr/"MD"/"mean_distance.py", mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))
md=mr/"MD"/"mean_distance.py"; s=md.read_text()
s=s.replace("all_dist = []","all_dist = []\n    import torch as _st; _st.manual_seed(42); _st.cuda.manual_seed_all(42)",1); md.write_text(s)
mm=pathlib.Path("/kaggle/temp/FreeFine/src/demo/model.py"); g=mm.read_text()
if not re.search(r'^\s*import os\b', g, re.M): g="import os\n"+g
assert "list(range(10, 16))" in g, "layer_idx hardcode missing"
n=g.count("list(range(10, 16))")
g=g.replace("list(range(10, 16))","list(range(int(os.environ.get('FF_START_LAYER','10')), 16))")
mm.write_text(g); print(f"patched metrics + model.py start_layer ({n} sites)")

patched metrics + model.py start_layer (5 sites)


In [5]:
# ===== C5 parametrize inference script (prompt + guidance + start_step + paths + ori_mask fix) =====
import os
P="/kaggle/temp/FreeFine/evaluation/FreeFine"; src=open(f"{P}/freefine_batch_infer_2d.py").read()
src=src.replace("sys.path.append('/data/Hszhu/FreeFine')","sys.path.append('/kaggle/temp/FreeFine')")
src=src.replace('pretrained_model_path = "/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/"','pretrained_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"')
old=('        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n        obj_label = ""\n        ori_mask = read_and_resize_mask(ori_mask_path)\n')
new=('        ori_mask = read_and_resize_mask(ori_mask_path)\n'
     '        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
     '        obj_label = (case.get("obj_label","") if os.environ.get("FF_USE_PROMPT")=="1" else "")\n')
assert old in src, "ori_mask/obj_label block mismatch"; src=src.replace(old,new,1)
src=src.replace('"guidance_scale": 7.5,','"guidance_scale": float(os.environ.get("FF_GUIDANCE","7.5")),')
src=src.replace('"start_step": 35,','"start_step": int(os.environ.get("FF_START_STEP","35")),')
src=src.replace('dataset_json = osp.join(dst_base, "annotations_2d.json")','dataset_json = os.environ.get("FF_SUBSET_JSON", osp.join(dst_base,"annotations_2d.json"))')
src=src.replace('dst_gen_dir = osp.join(dst_base, "Geo-Bench-2D/Gen_results_FreeFine_2d")','dst_gen_dir = os.environ.get("FF_OUT_DIR", osp.join(dst_base,"Geo-Bench-2D/Gen_results_FreeFine_2d"))')
src=src.replace('base_dir = "/data/Hszhu/dataset/GeoBenchMeta/"','base_dir = "/kaggle/temp/GeoBenchMeta"')
open(f"{P}/freefine_sweep_2d.py","w").write(src)
assert all(x in src for x in ["FF_USE_PROMPT","FF_GUIDANCE","FF_START_STEP"]), "parametrize failed"
print("parametrized: prompt + guidance + start_step (start_layer via model.py)")

parametrized: prompt + guidance + start_step (start_layer via model.py)


In [6]:
# ===== C5b B1 patches: AA-Warp + Adaptive-ES + save-coarse (on freefine_sweep_2d.py) =====
SW = "/kaggle/temp/FreeFine/evaluation/FreeFine/freefine_sweep_2d.py"
src = open(SW).read()
def patch(t, a, r, name):
    assert a in t, f"ANCHOR NOT FOUND: {name}"
    assert r not in t, f"ALREADY PATCHED: {name}"
    return t.replace(a, r, 1)

# P1: AA-Warp (supersampled Lanczos), gate FF_AA_WARP=1; mask warp untouched (stays NEAREST)
P1_A = "    transformed_image = cv2.warpAffine(src_img, rotation_matrix, (width, height))"
P1_R = """    if os.environ.get('FF_AA_WARP', '0') == '1':
        _S = 2
        _M = rotation_matrix.copy()
        _M[:, 2] *= _S
        _src_up = cv2.resize(src_img, (width * _S, height * _S), interpolation=cv2.INTER_LANCZOS4)
        _img_up = cv2.warpAffine(_src_up, _M, (width * _S, height * _S), flags=cv2.INTER_LANCZOS4)
        transformed_image = cv2.resize(_img_up, (width, height), interpolation=cv2.INTER_AREA)
    else:
        transformed_image = cv2.warpAffine(src_img, rotation_matrix, (width, height))"""
src = patch(src, P1_A, P1_R, "P1 AA-warp")

# P2a: difficulty LUT at module level (loads only when FF_META_CSV is set)
P2A_A = "def main(dst_base):"
P2A_R = """FF_DIFF_LUT = {}
_meta_csv = os.environ.get('FF_META_CSV', '')
if _meta_csv:
    import csv as _csv
    with open(_meta_csv) as _f:
        for _row in _csv.DictReader(_f):
            FF_DIFF_LUT[(str(_row['da_n']), str(_row['ins_id']), str(_row['case_id']))] = _row['difficulty']
    print(f'[B1] difficulty LUT loaded: {len(FF_DIFF_LUT)} cases', flush=True)
    assert len(FF_DIFF_LUT) == 5677, 'difficulty LUT incomplete'

def main(dst_base):"""
src = patch(src, P2A_A, P2A_R, "P2a LUT")

# P2b: per-case end_scale
P2B_A = "        edit_param = case['edit_param']"
P2B_R = """        edit_param = case['edit_param']
        if os.environ.get('FF_ES_MODE', 'fixed') == 'adaptive':
            _diff = FF_DIFF_LUT.get((str(da_n), str(ins_id), str(edit_ins)), 'medium')
            _ff_end_scale = {'easy': 0.0, 'medium': 0.25, 'hard': 0.5}[_diff]
        else:
            _ff_end_scale = float(os.environ.get('FF_END_SCALE', '0.0'))"""
src = patch(src, P2B_A, P2B_R, "P2b adaptive ES")

src = patch(src, '            "end_scale": 0.0,', '            "end_scale": _ff_end_scale,', "P2c wire-in")

# P3: save the (possibly AA) coarse so WRAP_E can use the right reference
P3_A = "        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)"
P3_R = """        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)
        if os.environ.get('FF_SAVE_COARSE', '0') == '1':
            _cdir = os.path.join(os.environ.get('FF_COARSE_DIR', '/kaggle/working/b1_coarse'), str(da_n), str(ins_id))
            os.makedirs(_cdir, exist_ok=True)
            Image.fromarray(coarse_input.astype(np.uint8)).save(os.path.join(_cdir, f"{edit_ins}.png"))"""
src = patch(src, P3_A, P3_R, "P3 save coarse")

open(SW, "w").write(src)
print("[B1] all 5 patches applied to freefine_sweep_2d.py")

[B1] all 5 patches applied to freefine_sweep_2d.py


In [7]:
# ===== C6 data + balanced-200 + coarse + reuse saved start_step variants =====
import os, glob, json, csv, random, shutil
from collections import defaultdict, Counter
GEO="/kaggle/temp/GeoBenchMeta"; os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)
CACHE=next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True) if os.path.isdir(f"{c}/source_img"))
COARSE=glob.glob("/kaggle/input/**/coarse_img/*/*/*.png",recursive=True)[0].split("/coarse_img/")[0]+"/coarse_img"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
IB=os.path.dirname(os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/inp_img_blended/**/inp_img.png",recursive=True)[0])))
ANNs=glob.glob("/kaggle/input/**/annotation_2d.json",recursive=True)[0]; META=glob.glob("/kaggle/input/**/sample_metadata.csv",recursive=True)[0]
VS=glob.glob("/kaggle/input/**/phase2/variants",recursive=True); VS=VS[0] if VS else None
for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}";  os.path.exists(d) or os.symlink(f"{CACHE}/{nm}",d)
for nm,sc in [("coarse_img",COARSE),("inp_img_blended",IB)]:
    d=f"{GEO}/Geo-Bench-2D/{nm}";  os.path.exists(d) or os.symlink(sc,d)
shutil.copy(ANNs,f"{GEO}/annotation_2d.json"); ann=json.load(open(f"{GEO}/annotation_2d.json"))
meta=[r for r in csv.DictReader(open(META)) if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]
random.seed(42); cells=defaultdict(list)
for r in meta: cells[(r["edit_type"],r["difficulty"])].append(r)
keys=sorted(cells); per=200//len(keys); picked=[]
for k in keys:
    pool=cells[k][:]; random.shuffle(pool); picked+=pool[:per]
ch={(r["da_n"],r["ins_id"],r["case_id"]) for r in picked}
left=[r for r in meta if (r["da_n"],r["ins_id"],r["case_id"]) not in ch]; random.shuffle(left)
for r in left:
    if len(picked)>=200: break
    picked.append(r)
picked=picked[:200]; json.dump(picked,open(f"{GEO}/subset_meta.json","w"))
print("subset",len(picked),dict(Counter(r["edit_type"] for r in picked)))
gsub={}
for r in picked:
    d,i,e=r["da_n"],r["ins_id"],r["case_id"]; lf=dict(ann[d]["instances"][i][e])
    lf["ori_img_path"]=os.path.join(GEO,lf["ori_img_path"]); lf["ori_mask_path"]=os.path.join(GEO,lf["ori_mask_path"])
    gsub.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
json.dump(gsub,open(f"{GEO}/gen_subset.json","w"))
os.makedirs(f"{GEO}/gen_eval",exist_ok=True); sets={"baseline":GENBASE}
if VS:
    for tag in ["ss25","ss30","ss40","ss45"]:
        if glob.glob(f"{VS}/{tag}/**/*.png",recursive=True): sets[tag]=f"{VS}/{tag}"
for nm,sc in sets.items():
    d=f"{GEO}/gen_eval/{nm}"
    if os.path.islink(d): os.remove(d)
    os.symlink(sc,d)
print("reused eval sets:",list(sets))

subset 200 {'move': 67, 'resize': 67, 'rotate': 66}
reused eval sets: ['baseline', 'ss25', 'ss30', 'ss40', 'ss45']


In [8]:
# ===== C7 validation gate (default params must reproduce baseline) =====
import os, json, socket, subprocess, glob, numpy as np
from PIL import Image
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"]=UserSecretsClient().get_secret("HF_TOKEN")
GEO="/kaggle/temp/GeoBenchMeta"; P="/kaggle/temp/FreeFine/evaluation/FreeFine"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
gs=json.load(open(f"{GEO}/gen_subset.json")); val={}; n=0
for d,da in gs.items():
    for i,ins in da["instances"].items():
        for e in ins:
            val.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=ins[e]; n+=1
            if n>=5:break
        if n>=5:break
    if n>=5:break
json.dump(val,open(f"{GEO}/val5.json","w")); os.makedirs("/kaggle/temp/val5",exist_ok=True)
env=os.environ.copy(); env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/val5.json","FF_OUT_DIR":"/kaggle/temp/val5"})
s=socket.socket();s.bind(("",0));port=s.getsockname()[1];s.close()
r=subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1","--master-port",str(port),"freefine_sweep_2d.py"],cwd=P,env=env,capture_output=True,text=True)
imgs=sorted(glob.glob("/kaggle/temp/val5/**/*.png",recursive=True))
if not imgs: print((r.stdout+r.stderr)[-3000:]); raise SystemExit("validation 0 images")
diffs=[float(np.abs(np.array(Image.open(n).convert("RGB"),float)-np.array(Image.open(f"{GENBASE}/{os.path.relpath(n,'/kaggle/temp/val5')}").convert("RGB").resize(Image.open(n).size),float)).mean()) for n in imgs]
print("validation mean|Δ|:",[round(x,3) for x in diffs]); assert max(diffs)<1.0,"NOT REPRODUCING"; print("✓ validation passed")

validation mean|Δ|: [0.0, 0.0, 0.0, 0.0, 0.0]
✓ validation passed


In [9]:
# ===== C8 generate NEW variants: start_layer (fixed) + prompt-enabled CFG =====
import os, time, glob, subprocess, socket, numpy as np
from PIL import Image
GEO="/kaggle/temp/GeoBenchMeta"; OUT="/kaggle/working/b1/variants"; os.makedirs(OUT,exist_ok=True)
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
GEN_DEADLINE=NB_START+6.5*3600
import glob as _g
META=_g.glob("/kaggle/input/**/sample_metadata.csv",recursive=True)[0]
NEW=[("aaw",    {"FF_AA_WARP":1,"FF_SAVE_COARSE":1,"FF_COARSE_DIR":"/kaggle/working/b1_coarse/aaw"}),
     ("es25",   {"FF_END_SCALE":0.25}),
     ("es50",   {"FF_END_SCALE":0.5}),
     ("esA",    {"FF_ES_MODE":"adaptive","FF_META_CSV":META}),
     ("aaw_esA",{"FF_AA_WARP":1,"FF_ES_MODE":"adaptive","FF_META_CSV":META,
                 "FF_SAVE_COARSE":1,"FF_COARSE_DIR":"/kaggle/working/b1_coarse/aaw_esA"})]
def gen2(o,**kw):
    os.makedirs(o,exist_ok=True); env=os.environ.copy()
    env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/gen_subset.json","FF_OUT_DIR":o})
    env.update({k:str(v) for k,v in kw.items()})
    s=socket.socket();s.bind(("",0));p=s.getsockname()[1];s.close()
    return subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=2","--master-port",str(p),"freefine_sweep_2d.py"],cwd="/kaggle/temp/FreeFine/evaluation/FreeFine",env=env,stdout=open(o+"/log.txt","w"),stderr=subprocess.STDOUT)
def pdiff(tag):
    ds=[]
    for n in sorted(glob.glob(f"{OUT}/{tag}/**/*.png",recursive=True))[:8]:
        rel=os.path.relpath(n,f"{OUT}/{tag}")
        ds.append(round(float(np.abs(np.array(Image.open(n).convert("RGB"),float)-np.array(Image.open(f"{GENBASE}/{rel}").convert("RGB").resize(Image.open(n).size),float)).mean()),2))
    return ds
done=[]
for tag,kw in NEW:
    if time.time()>GEN_DEADLINE: print("GEN_DEADLINE skip",tag,flush=True); break
    o=f"{OUT}/{tag}"; t=time.time()
    try:
        r=gen2(o,**kw); k=len(glob.glob(o+"/**/*.png",recursive=True))
        print(f"[{int((time.time()-NB_START)/60)}m] {tag}: rc={r.returncode} {int(time.time()-t)}s imgs={k} Δvs_baseline={pdiff(tag)}",flush=True)
        if k>0: done.append(tag)
        if r.returncode!=0: print("  tail:",open(o+'/log.txt').read()[-900:])
    except Exception as ex: print(f"{tag} FAILED:{ex}",flush=True)
print("new variants:",done,flush=True)

[83m] aaw: rc=0 4291s imgs=200 Δvs_baseline=[0.83, 0.2, 1.05, 0.83, 1.03, 0.19, 0.26, 1.18]
[155m] es25: rc=0 4311s imgs=200 Δvs_baseline=[0.47, 0.78, 0.33, 0.73, 0.48, 0.25, 0.33, 0.59]
[227m] es50: rc=0 4301s imgs=200 Δvs_baseline=[0.92, 1.55, 0.66, 1.53, 0.94, 0.5, 0.63, 1.21]
[299m] esA: rc=0 4316s imgs=200 Δvs_baseline=[0.0, 0.78, 0.0, 1.53, 0.48, 0.5, 0.0, 0.59]
[371m] aaw_esA: rc=0 4336s imgs=200 Δvs_baseline=[0.83, 0.85, 1.05, 1.93, 1.16, 0.59, 0.26, 1.35]
new variants: ['aaw', 'es25', 'es50', 'esA', 'aaw_esA']


In [10]:
# ===== C9 comprehensive eval — B1 sets (SUBC/BGC/WRAP_E all groups, MD on hard cells, FID-family @200) =====
import os, json, re, glob, subprocess, time, numpy as np
from PIL import Image
GEO="/kaggle/temp/GeoBenchMeta"; MET="/kaggle/temp/FreeFine/evaluation/metrics"; PY="/kaggle/temp/metric_env/bin/python"
EVAL_DEADLINE=NB_START+10.5*3600
ann=json.load(open(f"{GEO}/annotation_2d.json")); picked=json.load(open(f"{GEO}/subset_meta.json"))

# link the B1 variant outputs into gen_eval
for tag in ["aaw","es25","es50","esA","aaw_esA"]:
    p=f"/kaggle/working/b1/variants/{tag}"
    if glob.glob(p+"/**/*.png",recursive=True):
        d=f"{GEO}/gen_eval/{tag}"
        if os.path.islink(d): os.remove(d)
        os.symlink(p,d)

sets=[s for s in ["baseline","aaw","es25","es50","esA","aaw_esA"] if os.path.exists(f"{GEO}/gen_eval/{s}")]
print("eval sets:",sets)

def mem(pred): return [(r["da_n"],r["ins_id"],r["case_id"]) for r in picked if pred(r)]
groups={"rotate_hard":(mem(lambda r:r["edit_type"]=="rotate" and r["difficulty"]=="hard"),"000110100"),
        "resize_hard":(mem(lambda r:r["edit_type"]=="resize" and r["difficulty"]=="hard"),"000110100"),
        "move_all":(mem(lambda r:r["edit_type"]=="move"),"000110000"),
        "all_200":(mem(lambda r:True),"100110011")}

# WRAP_E with selectable coarse reference (AA variants must be scored against the AA coarse they used)
def wrap_e(ids,gd,coarse_base=None):
    cb = coarse_base or f"{GEO}/Geo-Bench-2D/coarse_img"
    tot=0.0;n=0
    for d,i,e in ids:
        cp=f"{cb}/{d}/{i}/{e}.png"; gp=f"{gd}/{d}/{i}/{e}.png"; tp=f"{GEO}/Geo-Bench-2D/target_mask/{d}/{i}/{e}.png"
        if not(os.path.exists(cp) and os.path.exists(gp) and os.path.exists(tp)): continue
        C=np.array(Image.open(cp).convert("RGB"),float)/255; G=np.array(Image.open(gp).convert("RGB"),float)/255; T=np.array(Image.open(tp).convert("L"),float)/255
        if G.shape[:2]!=C.shape[:2]: G=np.array(Image.fromarray((G*255).astype("uint8")).resize((C.shape[1],C.shape[0])),float)/255
        if T.shape[:2]!=C.shape[:2]: T=np.array(Image.fromarray((T*255).astype("uint8")).resize((C.shape[1],C.shape[0])),float)/255
        m=np.repeat(T[...,None],3,axis=2); su=m.sum()
        if su<=0: continue
        tot+=float(np.sum(np.abs(C*m-G*m))/su); n+=1
    return round(tot/n,4) if n else None

def manifest(setn,ids):
    o={}; b=f"{GEO}/gen_eval/{setn}"
    for d,i,e in ids:
        if not os.path.exists(f"{b}/{d}/{i}/{e}.png"): continue
        lf=dict(ann[d]["instances"][i][e]); lf["gen_img_path"]=f"gen_eval/{setn}/{d}/{i}/{e}.png"
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
    pth=f"{GEO}/m_{setn}_{len(ids)}.json"; json.dump(o,open(pth,"w")); return pth

def runm(manp,task):
    env=os.environ.copy(); env.update({"MPLBACKEND":"Agg","HF_HOME":"/kaggle/temp/hf","TORCH_HOME":"/kaggle/temp/torch","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True"})
    out=subprocess.run([PY,"main.py","--path",manp,"--use_relative_path","--base_dir",GEO,"--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task",task,"--level","0"],cwd=MET,env=env,capture_output=True,text=True)
    t=out.stdout+out.stderr; v={}
    for k in ["FID_DINO","FID_KD","FID","SUBC","BGC","MD"]:
        m=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",t)
        if m: v[k]=round(float(m[-1]),4)
    return v

# AA variants: primary WRAP_E vs the AA coarse they were generated from; also report vs original coarse
COARSE_OVR={"aaw":"/kaggle/working/b1_coarse/aaw","aaw_esA":"/kaggle/working/b1_coarse/aaw_esA"}

results={}; stop=False
for setn in sets:
    if stop: break
    results[setn]={}
    for g,(ids,task) in groups.items():
        if time.time()>EVAL_DEADLINE: print("EVAL_DEADLINE",flush=True); stop=True; break
        if not ids: continue
        v=runm(manifest(setn,ids),task)
        v["WRAP_E"]=wrap_e(ids,f"{GEO}/gen_eval/{setn}",COARSE_OVR.get(setn))
        if setn in COARSE_OVR:
            v["WRAP_E_vs_origcoarse"]=wrap_e(ids,f"{GEO}/gen_eval/{setn}")
        v["n"]=len(ids)
        results[setn][g]=v; print(f"{setn:8s} {g:12s} -> {v}",flush=True)
    json.dump(results,open("/kaggle/working/b1_full.json","w"),indent=2)
print("eval done")

eval sets: ['baseline', 'aaw', 'es25', 'es50', 'esA', 'aaw_esA']
baseline rotate_hard  -> {'SUBC': 0.8523, 'BGC': 0.9664, 'MD': 14.0096, 'WRAP_E': 0.0416, 'n': 22}
baseline resize_hard  -> {'SUBC': 0.8391, 'BGC': 0.9631, 'MD': 19.4713, 'WRAP_E': 0.0585, 'n': 23}
baseline move_all     -> {'SUBC': 0.959, 'BGC': 0.9639, 'WRAP_E': 0.0495, 'n': 67}
baseline all_200      -> {'FID_DINO': 1636.8248, 'FID_KD': 0.127, 'FID': 132.279, 'SUBC': 0.9154, 'BGC': 0.9657, 'WRAP_E': 0.0488, 'n': 200}
aaw      rotate_hard  -> {'SUBC': 0.8537, 'BGC': 0.9663, 'MD': 14.5439, 'WRAP_E': 0.0447, 'WRAP_E_vs_origcoarse': 0.0433, 'n': 22}
aaw      resize_hard  -> {'SUBC': 0.8403, 'BGC': 0.9628, 'MD': 19.6664, 'WRAP_E': 0.0573, 'WRAP_E_vs_origcoarse': 0.0587, 'n': 23}
aaw      move_all     -> {'SUBC': 0.9592, 'BGC': 0.9635, 'WRAP_E': 0.0454, 'WRAP_E_vs_origcoarse': 0.048, 'n': 67}
aaw      all_200      -> {'FID_DINO': 1637.268, 'FID_KD': 0.1297, 'FID': 132.3575, 'SUBC': 0.9156, 'BGC': 0.9654, 'WRAP_E': 0.0486, 'WRA

In [11]:
# ===== C10 summary =====
import json
R=json.load(open("/kaggle/working/b1_full.json"))
order=["baseline","aaw","es25","es50","esA","aaw_esA"]
for g in ["rotate_hard","resize_hard","move_all","all_200"]:
    print(f"\n=== {g} ===")
    cols=["SUBC","BGC","WRAP_E","WRAP_E_vs_origcoarse","MD","FID","FID_DINO","FID_KD"]
    print("set      "+"".join(f"{c:>22}" if c=="WRAP_E_vs_origcoarse" else f"{c:>10}" for c in cols))
    for s in order:
        if s in R and g in R[s]:
            d=R[s][g]; print(f"{s:8s} "+"".join(f"{str(d.get(c,'-')):>22}" if c=="WRAP_E_vs_origcoarse" else f"{str(d.get(c,'-')):>10}" for c in cols))
print("\nNotes: aaw/aaw_esA WRAP_E uses the AA coarse they were generated from;")
print("WRAP_E_vs_origcoarse = same gens scored against the original bilinear coarse (transparency).")
print("esA: easy tertile = baseline by design (end_scale 0.0/0.25/0.5 for easy/medium/hard).")
print("saved -> /kaggle/working/b1_full.json")


=== rotate_hard ===
set            SUBC       BGC    WRAP_E  WRAP_E_vs_origcoarse        MD       FID  FID_DINO    FID_KD
baseline     0.8523    0.9664    0.0416                     -   14.0096         -         -         -
aaw          0.8537    0.9663    0.0447                0.0433   14.5439         -         -         -
es25         0.8524    0.9671    0.0423                     -   14.7356         -         -         -
es50         0.8522    0.9672    0.0434                     -   14.9638         -         -         -
esA          0.8522    0.9672    0.0434                     -   14.9638         -         -         -
aaw_esA      0.8536    0.9662    0.0463                 0.045   15.0177         -         -         -

=== resize_hard ===
set            SUBC       BGC    WRAP_E  WRAP_E_vs_origcoarse        MD       FID  FID_DINO    FID_KD
baseline     0.8391    0.9631    0.0585                     -   19.4713         -         -         -
aaw          0.8403    0.9628    0.0573 